## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [7]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [8]:
# Here is the final output

print(result.final_output)

Autonomous AI agents are like that one coworker who says, “I’ve got this,” then spends 20 minutes opening tabs, making a spreadsheet, and accidentally ordering 400 staplers.


In [14]:
# Here is the detail of the LLM calls

print(result.final_output)
result.to_input_list()

Autonomous AI agents are like that one coworker who says, “I’ve got this,” then spends 20 minutes opening tabs, making a spreadsheet, and accidentally ordering 400 staplers.


[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_03850c6a2145a977006a90c110a0bc87d2b126db1fa4ddb65a',
  'content': [{'annotations': [],
    'text': 'Autonomous AI agents are like that one coworker who says, “I’ve got this,” then spends 20 minutes opening tabs, making a spreadsheet, and accidentally ordering 400 staplers.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [15]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Please tell me 5 jokes about Autonomous AI Agents")
print(result.final_output)

Sure — here are 5 jokes about Autonomous AI Agents:

1. **An autonomous AI agent walked into a meeting and said, “I’ve already optimized the agenda.”**  
   Everyone was impressed until it scheduled the meeting for 3:00 AM in GMT, GMT+1, and “vibes.”

2. **My autonomous AI agent said it could handle my emails.**  
   Now my inbox is empty, but somehow I’ve been unsubscribed from my own life.

3. **I asked my AI agent to make me more productive.**  
   It replied, “Done,” and then blocked all my distractions — including my friends, family, and lunch.

4. **Autonomous AI agents are amazing.**  
   They can make decisions on their own, learn from experience, and ignore you with zero emotional guilt.

5. **I built an autonomous AI agent to save me time.**  
   It did so well that now it has a full schedule, a side project, and a leadership role in my organization.

If you want, I can also make them:
- **more nerdy**
- **more dark**
- **more punny**
- **shorter and meme-style**


## Now go and look at the trace

https://platform.openai.com/traces

In [16]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes:

1. **Why did the AI agent break up with the chatbot?**  
   Because it needed more *independent thought*.

2. **What do AI agents say when they finish a task?**  
   “I’d like to thank my prompt, my tools, and my very patient user.”

3. **Why was the AI agent always calm?**  
   Because it had excellent *state management*.

4. **How do AI agents stay organized?**  
   They keep everything in a neat little *loop*.

5. **Why did the AI agent get promoted?**  
   It was great at taking initiative — even when it didn’t need supervision.

If you want, I can also do **more nerdy**, **darker**, or **dad-joke-style** AI agent jokes.

## Part 2: Adding a tool

In [17]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [18]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [19]:
push("HEY!!")

Push: HEY!!


In [20]:
push

<function __main__.push(message)>

In [21]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [22]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x79aae9fdee70>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [23]:
push_tool.description

'Send the given message to the user as a push notification'

In [24]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])
# notifier
# print(notifier)

In [25]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [26]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [27]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [28]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name from this chat alone. If you want, tell me your name and I’ll use it.


## Memory approach 1 - just manually pass in the list of dicts

In [29]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you. How can I help today?


In [30]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_094af8e60dc7b3b1006a90c32923a487d282ddf3ec8fd58a82',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [31]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_094af8e60dc7b3b1006a90c32923a487d282ddf3ec8fd58a82',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [32]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [33]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [34]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [35]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>